There's already a website with every cards information, which we can retrieve for future use.

Currently, when doing a Performance Comparison, vanilla triggers or sentinels will create lots of outliers during analysis. 
It would be much easier if we could just check if a card is a vanilla trigger, or in a future stage of analysis, look for things of note in card text.

Given that we will want to look up this information often, it would be wise to store the data in a database so we don't need to do thousands of web lookups per analysis. Instead, we can do thousands of web lookups once, and maybe a few more every now and again, when the data doesn't appear in the database.



Since different cards can share a name, and card's can have effects that make them share names, or have extra names; we should include that information in a way for easy lookups, and avoid using card name's as ID's

We're already using a NOSQL database, so having a dictionary of card names for each card seems appropriate

idk how the website will be strucuted, so we should investigate skipping this step, but I know decklog has a link to each cards page, so as a last resort, we can take a decklog as input with out lookup function (form whicher analysis finds we're missing a cards info, while looking over a particular deck, we can just pass the decklog), and then find the cards section of the page, and follow the link. This seems awkward, but it is potentially reliable.

In [ ]:

import requests


import db_operations


from bs4 import BeautifulSoup as Soup

In [18]:
URLS = [
#Benerator
    'https://en.cf-vanguard.com/cardlist/?cardno=BCS2425/VGS04',

    # BlessFavor (trigger unit)
'https://en.cf-vanguard.com/cardlist/?cardno=DZ-BT07/024EN',

#     # Yuika (normal unit)
# # # URL = 
'https://en.cf-vanguard.com/cardlist/?cardno=DZ-SS08/Re49EN',

# Bav (g3)
'https://en.cf-vanguard.com/cardlist/?cardno=DZ-BT07/001EN',

# PG
'https://en.cf-vanguard.com/cardlist/?cardno=DZ-LBT02/FR42EN',

#     # Stil Nacht (blitz)
#  # # URL = 
'https://en.cf-vanguard.com/cardlist/?cardno=DZ-LBT02/019EN',

#     # Silent Snowfall
# # # URL = 
'https://en.cf-vanguard.com/cardlist/?cardno=DZ-BT06/025EN',

#     # Mygo music
# # # URL = 
'https://en.cf-vanguard.com/cardlist/?cardno=DZ-BT01/EX41EN',

#     # Buddy Fight Trigger order
# # # URL = 
'https://en.cf-vanguard.com/cardlist/?cardno=DZ-TB01/077EN'
]
data_list = []
for url in URLS:
    request = requests.get(url)
    soup=Soup(request.text, 'html.parser')
    data = soup.find(attrs={'class':'data'})
    data_list.append(data)

In [19]:
for i, data in enumerate(data_list):
    print(type(data))

<class 'bs4.element.Tag'>
<class 'bs4.element.Tag'>
<class 'bs4.element.Tag'>
<class 'bs4.element.Tag'>
<class 'bs4.element.Tag'>
<class 'bs4.element.Tag'>
<class 'bs4.element.Tag'>
<class 'bs4.element.Tag'>
<class 'bs4.element.Tag'>


In [20]:
bar = []
for data in data_list:
    foo = []
    for el in data:
        text = el.text
        if text.strip() != '':
            foo.append(text.strip())
    bar.append(foo)

# for bar in foo:
#     bar = bar.split('\n')
#     print(bar)
#     print('~~~~~')

bar

[['Energy Generator',
  'Ride Deck Crest\n-\n-\nGrade -\nPower \nCritical \nShield',
  '(You may only have one ride deck crest in a ride deck)\r\n[AUTO]Ride Deck:When you ride, put this card into the crest zone, and if you went second, [Energy-Charge 3].\r\n[CONT]:You may have up to ten energy.\r\n[AUTO]:At the beginning of your ride phase, [Energy-Charge 3].\r\n[ACT][1/turn]:[COST][Energy-Blast 7], and draw a card.',
  'Standard\nBCS2425/VGS04\nPR\n©VANGUARD Divinez\u3000Character Design ©2021-2024 CLAMP・ST\u3000illust:Kinema citrus'],
 ['Source Dragon Deity of Blessings, Blessfavor',
  'Trigger Unit\nStoicheia\nNature Dragon\nGrade 0\nPower 5000\nCritical 1\nShield 50000\nBoost\nOver Trigger +100,000,000',
  "(You may only have one [Over] in a deck. When revealed as a trigger, remove that card, draw a card, choose one of your units, and it gets [Power] +100 Million until end of turn! If revealed during drive check, activate its additional effect!)Additional Effect - Draw a card! Choo

In [13]:
for foo in bar:
    for baz in foo:
        print(baz.split('\n'))
        print('~~~~~~~~~~~~~~~~~`')

['Energy Generator']
~~~~~~~~~~~~~~~~~`
['Ride Deck Crest', '-', '-', 'Grade -', 'Power ', 'Critical ', 'Shield']
~~~~~~~~~~~~~~~~~`
['(You may only have one ride deck crest in a ride deck)\r', '[AUTO]Ride Deck:When you ride, put this card into the crest zone, and if you went second, [Energy-Charge 3].\r', '[CONT]:You may have up to ten energy.\r', '[AUTO]:At the beginning of your ride phase, [Energy-Charge 3].\r', '[ACT][1/turn]:[COST][Energy-Blast 7], and draw a card.']
~~~~~~~~~~~~~~~~~`
['Standard', 'BCS2425/VGS04', 'PR', '©VANGUARD Divinez\u3000Character Design ©2021-2024 CLAMP・ST\u3000illust:Kinema citrus']
~~~~~~~~~~~~~~~~~`


In [32]:

def extract_data_from_html(html):
    data_entry = dict()
    test = []
        
    for i, item in enumerate(html):
        if i == 2|3: test.append(item) # We want to preserve line breaks in flavor text?
        else: test.append(item.split('\n'))

    card_type = test[1][0]

    data_entry['name'] = test[0][0]

    data_entry['type'] = card_type
    data_entry['nation'] = test[1][1]
    data_entry['race'] = test[1][2]
    
    if "Crest" in card_type:
        data_entry['grade'] = "None"
    else:
        data_entry['grade'] = int(test[1][3].split(' ')[1])

    if "Unit" in card_type:
        data_entry['power'] = int(test[1][4].split(' ')[1])
        data_entry['crit'] = int(test[1][5].split(' ')[1])
        # Grade 3's have an empty string when checking for shield, so this fixes that bug
        shield = test[1][6].split(' ')[1]
        if shield != '':
            data_entry['shield'] = int(shield)
        else:
            data_entry['shield'] = 0

        data_entry['ability'] = test[1][7]
    else:
        data_entry['power'] = "None"
        data_entry['crit'] = "None"
        data_entry['shield'] = "None"
        data_entry['ability'] = "None"

    if "Trigger" in card_type:
        data_entry['trigger'] = test[1][8]
    else:
        data_entry['trigger'] = "None"

    data_entry['effect'] = test[2]

    if len(test) == 4:
        test.insert(3, '')

    data_entry['flavor'] = test[3]

    data_entry['format'] = test[4][0]
    data_entry['id'] = test[4][1]
    data_entry['rarity'] = test[4][2]
    data_entry['artist'] = test[4][3]

    return data_entry

In [33]:
for foo in bar:
    print(extract_data_from_html(foo)['effect'])

['(You may only have one ride deck crest in a ride deck)\r', '[AUTO]Ride Deck:When you ride, put this card into the crest zone, and if you went second, [Energy-Charge 3].\r', '[CONT]:You may have up to ten energy.\r', '[AUTO]:At the beginning of your ride phase, [Energy-Charge 3].\r', '[ACT][1/turn]:[COST][Energy-Blast 7], and draw a card.']
["(You may only have one [Over] in a deck. When revealed as a trigger, remove that card, draw a card, choose one of your units, and it gets [Power] +100 Million until end of turn! If revealed during drive check, activate its additional effect!)Additional Effect - Draw a card! Choose one of your units, and it gets [Critical] +1 until end of turn! All of your front row units get [Power] +10000! If your damage zone has the same number of cards as your opponent's or more, choose a card from your damage zone, and heal it!"]
["[AUTO](RC)[1/Turn]:When your other unit is placed in the same column as this unit, if your opponent's vanguard is grade 3 or grea

In [ ]:



def add_card_info_to_db(url: str) -> bool:
    """Given a card name, add its information to a central data base"""

    try:
        html_text = get_card_info_from_bushi_site(url)

        data_entry = exctract_info_from_bushi_site(html_text)

        db_operations.insert_one_into_table('main_table',
                                            TODO,
                                            data_entry)

        return True

    except Exception as something_went_wrong:
        print(something_went_wrong)
        return False


def get_card_info_from_bushi_site(url: str) -> list[list[str]]:
    """Given a name, and maybe decklog, retrieve the text of the official webpage with it's information
    
    The wepbage contains an element, 'data', which has all the informatin about the card for the URL.
    It contains some sub elements, so the easiest way to extract it is to simply retrieve the text from each,
    and put it into a big list.

    There are lots of different atributes of the card that are contained in one element, seperated by `\n`
    new line charachters. So, for each element, we'll turn it into a list of attributes,
    ending us with a list of lists of string. list [ list [ str ] ]

    # name 
    # stats
    # effect 
    # flavor
    # bottomline(set, artist, etc.)

    These are the 5 children of the data element, each with a different number of attributes.
    """
    response = requests.get(url)
    soup=Soup(response.text, 'html.parser')
    data_element = soup.find(attrs={'class':'data'})

    data = []
    for child in data_element:
        text = child.text
        if text.strip() != '':
            data.append(text.strip())

    return data


def exctract_info_from_bushi_site(seperated_soup: list[list[str]]) -> dict:
    """Given the text from a card's bushi site page, return the information we want in our database
    
    We hit a few snags, due to the combinations of order, trigger, and unit atributes that can cause
    some headache.

    For now, Order type (Music, Codex, Fox Art, Etc.) are placed in the 'race' section,
    and orders with no type just have '-' as their race. We can wrange these issues later.
    At least we have the data.
    """

    data_entry = dict()
    test = []
    for i, item in enumerate(seperated_soup):
        if i == 3: test.append(item) # We want to preserve line breaks in flavor text?
        else: test.append(item.split('\n'))

    card_type = test[1][0]

    data_entry['name'] = test[0][0]

    data_entry['type'] = card_type
    data_entry['nation'] = test[1][1]
    data_entry['race'] = test[1][2]
    data_entry['grade'] = int(test[1][3].split(' ')[1])
    # ~~~~~~~~~~~~~~~~~~   UNIT / ORDER / TRIGGER LOGIC ~~~~~~~~~~~~~~~~~~~~~~~
    if "Unit" in card_type:
        data_entry['power'] = int(test[1][4].split(' ')[1])
        data_entry['crit'] = int(test[1][5].split(' ')[1])
        # Grade 3's have an empty string when checking for shield, so this fixes that bug
        shield = test[1][6].split(' ')[1]
        if shield == '':
            data_entry['shield'] = 0
        else:
            data_entry['shield'] = int(shield)

        data_entry['ability'] = test[1][7]

    else:
        data_entry['power'] = "None"
        data_entry['crit'] = "None"
        data_entry['shield'] = "None"
        data_entry['ability'] = "None"

    if "Trigger" in card_type:
        data_entry['trigger'] = test[1][8]
    else:
        data_entry['trigger'] = "None"

    data_entry['effect'] = test[2]
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    # Some cards don't have flavor text, and this 'Magic Number Hack' inserts an empty string
    if len(test) == 4:
        test.insert(3, '')
    data_entry['flavor'] = test[3]

    data_entry['format'] = test[4][0]
    data_entry['id'] = test[4][1]
    data_entry['rarity'] = test[4][2]
    data_entry['illust'] = test[4][3]

    return data_entry
